# AeroPass: trening modelu zalanych dróg (FloodNet, YOLO11-seg)

**Jak uruchomić:** *Środowisko wykonawcze → Zmień typ środowiska → T4 GPU*, potem *Środowisko wykonawcze → Uruchom wszystko*. Całość trwa ok. 40–60 min. Na końcu przeglądarka pobierze `wyniki_aeropass.zip`; ten plik trzeba przekazać Claude'owi.

- Dane: FloodNet-Supervised v1.0 (CDLA-Permissive 1.0), **oficjalny podział train/val/test**.
- Model: Ultralytics YOLO11n-seg (**AGPL-3.0**).
- Klasy: 0 zalana droga, 1 zalany budynek, 2 niezalana droga.
- Główna metryka decyzji: **odsetek fałszywie bezpiecznych**, czyli ile zalanych dróg model uznał za przejezdne. Liczymy go na zbiorze testowym.
- **Nigdzie nie wpisujemy liczb, których nie zmierzyliśmy.**

In [ ]:
# 1. Instalacja i GPU
!pip install -q ultralytics
import torch, ultralytics
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "BRAK: zmień typ środowiska na T4")
print("ultralytics", ultralytics.__version__)
assert torch.cuda.is_available(), "Włącz GPU: Środowisko wykonawcze → Zmień typ środowiska → T4 GPU"

In [ ]:
# 2. Pobranie i rozpakowanie FloodNet (13 GB; ok. 10–20 min)
!wget -q --show-progress -O /content/FloodNet.zip "https://www.dropbox.com/scl/fo/k33qdif15ns2qv2jdxvhx/ANGaa8iPRhvlrvcKXjnmNRc?rlkey=ao2493wzl1cltonowjdbrnp7f&dl=1"
!unzip -q /content/FloodNet.zip -d /content/FloodNet || 7z x -y /content/FloodNet.zip -o/content/FloodNet > /dev/null
!rm -f /content/FloodNet.zip
!find /content/FloodNet -maxdepth 3 -type d

In [ ]:
# 2b. Kontrola struktury i numeracji klas (zatrzyma notatnik, jeśli coś się nie zgadza)
import json
from collections import Counter
from pathlib import Path
import numpy as np
from PIL import Image

ROOT = Path("/content")
SRC = ROOT / "FloodNet"
m_all = sorted(p for p in SRC.rglob("*_lab.png") if "label-img" in str(p.parent))  # bez ColorMasks
print("masek *_lab.png:", len(m_all), "przykład:", m_all[:1])
assert m_all, "Nie ma masek *_lab.png: sprawdź wynik find w komórce 2"
hist = Counter()
for m in m_all[::10]:  # co dziesiąta maska ze wszystkich zbiorów (zalane drogi są rzadkie)
    hist.update(dict(enumerate(np.bincount(np.asarray(Image.open(m)).ravel(), minlength=11).tolist())))
hist = {k: v for k, v in sorted(hist.items()) if v}
print("piksele klas w próbce:", hist)
OFFSET = 1 if (0 not in hist and max(hist) == 10) else 0
print("OFFSET =", OFFSET, "(0: klasy 0–9, 1: klasy 1–10)")
assert (3 + OFFSET) in hist and (4 + OFFSET) in hist, "W próbce brak klas zalanej/niezalanej drogi: sprawdź maski"

In [ ]:
# 3. Konwersja do YOLO-seg: oficjalny podział z folderów, maski <id>_lab.png, obrazy zmniejszone do 640 px (ok. 10 min)
import cv2
DEST, SZER = ROOT / "yolo_dataset", 640
MAP_KLAS = {3 + OFFSET: 0, 1 + OFFSET: 1, 4 + OFFSET: 2}
NAZWY = {0: "flooded_road", 1: "flooded_building", 2: "road_non_flooded"}

def split_z_sciezki(p):
    # bez losowania po klatkach: sąsiednie ujęcia w treningu i teście zawyżają wynik
    return next((s for part in p.relative_to(SRC).parts for s in ("train", "val", "test") if part.lower().startswith(s)), None)

def poligony(mask):
    h, w = mask.shape[:2]
    linie = []
    for fn_cls, y_cls in MAP_KLAS.items():
        b = (mask == fn_cls).astype(np.uint8)
        if not b.any():
            continue
        cs, _ = cv2.findContours(b, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for c in cs:
            if cv2.contourArea(c) < 60:
                continue
            p = cv2.approxPolyDP(c, 0.005 * cv2.arcLength(c, True), True).reshape(-1, 2)
            if len(p) >= 3:
                linie.append(f"{y_cls} " + " ".join(f"{x / w:.5f} {y / h:.5f}" for x, y in p))
    return linie

for s in ("train", "val", "test"):
    (DEST / "images" / s).mkdir(parents=True, exist_ok=True)
    (DEST / "labels" / s).mkdir(parents=True, exist_ok=True)
maski = {(split_z_sciezki(m), m.stem.removesuffix("_lab")): m for m in SRC.rglob("*.png") if "label-img" in str(m.parent)}
pary = {"val": [], "test": []}
ile = {"train": 0, "val": 0, "test": 0}
for img in SRC.rglob("*.jpg"):
    s = split_z_sciezki(img)
    m = maski.get((s, img.stem))
    if s is None or m is None:
        continue
    (DEST / "labels" / s / f"{img.stem}.txt").write_text("\n".join(poligony(np.array(Image.open(m)))) + "\n")
    im = Image.open(img)
    im.resize((SZER, SZER * im.height // im.width), Image.BILINEAR).save(DEST / "images" / s / img.name)
    if s in pary:
        pary[s].append([str(DEST / "images" / s / img.name), str(m)])
    ile[s] += 1
assert ile["train"] > 0 and ile["val"] > 0, f"Za mało par: {ile}"
json.dump(pary, open(ROOT / "pary.json", "w"))
print("pary zdjęcie–maska:", ile)

In [ ]:
# 4. Konfiguracja zbioru
import yaml
cfg = {"path": str(DEST), "train": "images/train", "val": "images/val", "names": NAZWY}
if ile["test"]:
    cfg["test"] = "images/test"
yaml.safe_dump(cfg, open(DEST / "data.yaml", "w"), allow_unicode=True)
print(open(DEST / "data.yaml").read())

In [ ]:
# 5. Trening (ok. 15–30 min na T4)
from ultralytics import YOLO
EPOKI = 25
model = YOLO("yolo11n-seg.pt")
model.train(data=str(DEST / "data.yaml"), epochs=EPOKI, imgsz=640, batch=16, device=0,
            project=str(ROOT / "runs"), name="aeropass", exist_ok=True)

In [ ]:
# 6. Metryki masek na zbiorze walidacyjnym (tylko zmierzone wartości)
import shutil
best = YOLO(str(ROOT / "runs/aeropass/weights/best.pt"))
met = best.val(data=str(DEST / "data.yaml"), split="val")
W = ROOT / "wyniki"
W.mkdir(exist_ok=True)
podsum = {
    "model": f"YOLO11n-seg, Ultralytics {ultralytics.__version__} (AGPL-3.0)",
    "dane": "FloodNet-Supervised v1.0, oficjalny podział, CDLA-Permissive 1.0",
    "liczba_par": ile, "epoki": EPOKI, "imgsz": 640, "offset_klas": OFFSET,
    "val_mAP50_maski": float(met.seg.map50),
    "val_mAP50_95_maski": float(met.seg.map),
}
try:
    podsum["val_mAP50_95_maski_per_klasa"] = {NAZWY[i]: float(met.seg.maps[i]) for i in NAZWY}
except Exception as e:
    podsum["val_mAP50_95_maski_per_klasa"] = f"niedostępne: {e}"
json.dump(podsum, open(W / "metryki.json", "w"), indent=2, ensure_ascii=False)
for f in ("results.png", "confusion_matrix.png"):
    if (ROOT / "runs/aeropass" / f).exists():
        shutil.copy(ROOT / "runs/aeropass" / f, W / f)
print(json.dumps(podsum, indent=2, ensure_ascii=False))

In [ ]:
# 7. Metryka decyzji: odsetek fałszywie bezpiecznych (jednostka: zdjęcie jako przybliżenie odcinka drogi)
PROG_ZALANY, PROG_SUCHY, PROG_GT = 0.4, 0.5, 0.01

def stan(wykrycia):  # ta sama logika co ai/yolo_to_decision.py: brak dowodu = "nieznany"
    zal = [d["conf"] for d in wykrycia if d["name"] == "flooded_road" and d["conf"] >= PROG_ZALANY]
    suc = [d["conf"] for d in wykrycia if d["name"] == "road_non_flooded" and d["conf"] >= PROG_SUCHY]
    if zal:
        return {"stan": "zalany", "pewnosc": round(max(zal), 2)}
    if suc:
        return {"stan": "przejezdny", "pewnosc": round(max(suc), 2)}
    return {"stan": "nieznany", "pewnosc": 0.0}

def gt(mask_path):
    m = np.asarray(Image.open(mask_path))
    zal, suc = (m == 3 + OFFSET).mean(), (m == 4 + OFFSET).mean()
    return "zalany" if zal >= PROG_GT else ("przejezdny" if suc >= PROG_GT else None)

def wykryj(img):
    r = best.predict(img, imgsz=640, conf=0.25, verbose=False)[0]
    return r, [{"name": best.names[int(c)], "conf": round(float(p), 3)} for c, p in zip(r.boxes.cls, r.boxes.conf)]

SPLIT = "test" if pary["test"] else "val"
tab = Counter()
for img, mask in pary[SPLIT]:
    g = gt(mask)
    if g is not None:
        tab[(g, stan(wykryj(img)[1])["stan"])] += 1
n_zal = sum(v for (g, _), v in tab.items() if g == "zalany")
n_suc = sum(v for (g, _), v in tab.items() if g == "przejezdny")
dziel = lambda a, b: round(a / b, 4) if b else None
wynik = {
    "zbior": SPLIT + (" (UWAGA: brak etykiet testowych, użyto walidacyjnego)" if SPLIT == "val" else ""),
    "jednostka": "zdjęcie (przybliżenie odcinka drogi)",
    "definicja_prawdy": f"zalany: ≥{PROG_GT:.0%} pikseli to zalana droga; przejezdny: ≥{PROG_GT:.0%} to niezalana droga i mniej niż {PROG_GT:.0%} zalanej",
    "progi_pewnosci": {"zalany": PROG_ZALANY, "suchy": PROG_SUCHY},
    "liczba_zdjec": {"zalany": n_zal, "przejezdny": n_suc},
    "odsetek_falszywie_bezpiecznych": dziel(tab[("zalany", "przejezdny")], n_zal),
    "czulosc_wykrycia_zalania": dziel(tab[("zalany", "zalany")], n_zal),
    "zalane_oznaczone_jako_nieznane": dziel(tab[("zalany", "nieznany")], n_zal),
    "falszywe_alarmy_na_suchych": dziel(tab[("przejezdny", "zalany")], n_suc),
    "suche_oznaczone_jako_nieznane": dziel(tab[("przejezdny", "nieznany")], n_suc),
    "macierz": {f"{g} -> {p}": v for (g, p), v in sorted(tab.items())},
}
json.dump(wynik, open(W / "metryki_decyzji.json", "w"), indent=2, ensure_ascii=False)
print(json.dumps(wynik, indent=2, ensure_ascii=False))

In [ ]:
# 8. Przykłady do demo: 20 zalanych i 20 suchych zdjęć z predykcjami (także błędy)
D = W / "demo"
D.mkdir(exist_ok=True)
z_gt = [(i, m, gt(m)) for i, m in pary[SPLIT]]
wybrane = [x for x in z_gt if x[2] == "zalany"][:20] + [x for x in z_gt if x[2] == "przejezdny"][:20]
demo = []
for img, mask, g in wybrane:
    r, w = wykryj(img)
    cv2.imwrite(str(D / Path(img).name), r.plot())
    demo.append({"plik": f"demo/{Path(img).name}", "prawda": g, **stan(w), "wykrycia": w})
json.dump(demo, open(W / "predykcje_demo.json", "w"), indent=1, ensure_ascii=False)
print(len(demo), "przykładów;", sum(d["prawda"] == "zalany" and d["stan"] == "przejezdny" for d in demo), "fałszywie bezpiecznych wśród nich")

In [ ]:
# 9. Spakowanie i pobranie wyników
shutil.copy(ROOT / "runs/aeropass/weights/best.pt", W / "best.pt")
!cd /content && zip -qr wyniki_aeropass.zip wyniki
from google.colab import files
files.download("/content/wyniki_aeropass.zip")